# Module 7 — Target-Specific Normalization + ATCNet/DAFM Probability Ensemble

Final practical ensemble for the prepared 3-class BCI-IV-2a cache.

Input: `(N,22,640)` at 160 Hz.
Classes: left / right / feet.
Outer evaluation: BCI-IV-2a S01–S09 LOSO.

Two independently trained model families are combined:
- ATCNet-3C
- DAFM-3C

For the held-out target subject, normalization is estimated from that subject's **unlabeled EEG only**. Target labels are used only after predictions are generated.

The fusion weight is selected from SOURCE validation predictions only.

In [1]:
# ============================================================
# CELL 1 — IMPORTS / REPRODUCIBILITY
# ============================================================

from __future__ import annotations

import os
import gc
import copy
import time
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader,
    WeightedRandomSampler,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

SEED = 42

def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything()

device = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

print("Device:", device)

Device: cpu


In [2]:
# ============================================================
# CELL 2 — EXISTING CACHE
# ============================================================

PROJECT_ROOT = Path(
    "/Users/ashokvarmabevara/Project2"
)

PROJECT_DIR = (
    PROJECT_ROOT
    / "cross_dataset_mi_project"
)

CACHE_DIR = (
    PROJECT_DIR
    / "cache"
)

RESULT_DIR = (
    PROJECT_DIR
    / "results"
    / "module_7_atcnet_dafm"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CACHE_PATH = (
    CACHE_DIR
    / "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
)

if not CACHE_PATH.exists():

    candidates = sorted(
        CACHE_DIR.glob("*.h5")
    )

    if not candidates:
        raise FileNotFoundError(
            f"No HDF5 files in {CACHE_DIR}"
        )

    preferred = [
        p for p in candidates
        if "160hz" in p.name.lower()
    ]

    CACHE_PATH = (
        preferred[0]
        if preferred
        else candidates[0]
    )

print(
    "Cache:",
    CACHE_PATH,
)

assert CACHE_PATH.exists()

Cache: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5


In [3]:
# ============================================================
# CELL 3 — METADATA
# ============================================================

def decode(v):
    return (
        v.decode("utf-8")
        if isinstance(v, bytes)
        else str(v)
    )


with h5py.File(
    CACHE_PATH,
    "r",
) as h5:

    X_shape = tuple(
        h5["X"].shape
    )

    X_dtype = str(
        h5["X"].dtype
    )

    meta = {}

    for key in [
        "dataset",
        "subject",
        "run",
        "recording_id",
        "filename",
        "absolute_path",
        "harmonized_class",
    ]:

        meta[key] = [
            decode(v)
            for v in h5[
                "metadata"
            ][key][:]
        ]


cache_meta_df = pd.DataFrame(
    meta
)

cache_meta_df.insert(
    0,
    "cache_index",
    np.arange(
        len(cache_meta_df),
        dtype=np.int64,
    ),
)

CLASSES = [
    "left",
    "right",
    "feet",
]

CLASS_TO_ID = {
    c: i
    for i, c in enumerate(CLASSES)
}

N_CLASSES = 3

assert X_shape[1:] == (
    22,
    640,
)

assert X_dtype == "float32"

bci_meta = cache_meta_df[
    cache_meta_df[
        "dataset"
    ].astype(str)
    == "BCI-IV-2a"
].copy()

bci_meta["subject"] = (
    bci_meta["subject"].astype(str)
)

BCI_SUBJECTS = sorted(
    bci_meta[
        "subject"
    ].unique()
)

assert len(BCI_SUBJECTS) == 9

print(
    "Cache shape:",
    X_shape,
)

print(
    "Subjects:",
    BCI_SUBJECTS,
)

print(
    "Trials:",
    len(bci_meta),
)

Cache shape: (9316, 22, 640)
Subjects: ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09']
Trials: 1944


In [4]:
# ============================================================
# CELL 4 — DATA LOADER
# ============================================================

def load_indices(
    indices,
):

    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    with h5py.File(
        CACHE_PATH,
        "r",
    ) as h5:

        X = np.asarray(
            h5["X"][indices],
            dtype=np.float32,
        )

    return X


print(
    "✅ HDF5 loader ready."
)

✅ HDF5 loader ready.


In [5]:
# ============================================================
# CELL 5 — SOURCE NORMALIZATION
# ============================================================

class RobustNormalizer:

    def __init__(
        self,
        eps=1e-6,
    ):
        self.eps = eps
        self.median_ = None
        self.iqr_ = None

    def fit(
        self,
        X,
    ):

        X = np.asarray(
            X,
            dtype=np.float32,
        )

        V = (
            X
            .transpose(
                1,
                0,
                2,
            )
            .reshape(
                X.shape[1],
                -1,
            )
        )

        self.median_ = np.median(
            V,
            axis=1,
        )

        q25 = np.percentile(
            V,
            25,
            axis=1,
        )

        q75 = np.percentile(
            V,
            75,
            axis=1,
        )

        self.iqr_ = np.maximum(
            q75 - q25,
            self.eps,
        )

        return self

    def transform(
        self,
        X,
    ):

        if self.median_ is None:
            raise RuntimeError(
                "Normalizer not fitted."
            )

        X = np.asarray(
            X,
            dtype=np.float32,
        )

        Z = (
            X
            - self.median_[
                None,
                :,
                None,
            ]
        ) / (
            self.iqr_[
                None,
                :,
                None,
            ]
            + self.eps
        )

        return np.nan_to_num(
            Z,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(
            np.float32
        )


def fit_target_normalizer(
    X_target,
):
    """
    Unlabeled target-specific robust normalization.
    """

    normalizer = (
        RobustNormalizer()
        .fit(
            X_target
        )
    )

    return (
        normalizer.transform(
            X_target
        ),
        normalizer,
    )

In [6]:
# ============================================================
# CELL 6 — SOURCE TRIAL SPLIT
# ============================================================

def stratified_split(
    y,
    val_fraction=0.20,
    seed=SEED,
):

    y = np.asarray(
        y,
        dtype=np.int64,
    )

    rng = np.random.default_rng(
        seed
    )

    all_idx = np.arange(
        len(y)
    )

    train_parts = []
    val_parts = []

    for cls in range(
        N_CLASSES
    ):

        cls_idx = all_idx[
            y == cls
        ].copy()

        rng.shuffle(
            cls_idx
        )

        n_val = max(
            1,
            int(
                round(
                    len(cls_idx)
                    * val_fraction
                )
            ),
        )

        val_parts.append(
            cls_idx[
                :n_val
            ]
        )

        train_parts.append(
            cls_idx[
                n_val:
            ]
        )

    train_idx = np.concatenate(
        train_parts
    )

    val_idx = np.concatenate(
        val_parts
    )

    rng.shuffle(
        train_idx
    )

    rng.shuffle(
        val_idx
    )

    return (
        train_idx,
        val_idx,
    )

In [7]:
# ============================================================
# CELL 7 — ATCNET TCN + FRONTEND
# ============================================================

class CausalConv1d(
    nn.Module
):

    def __init__(
        self,
        in_ch,
        out_ch,
        kernel_size,
        dilation=1,
    ):

        super().__init__()

        self.pad = (
            kernel_size - 1
        ) * dilation

        self.conv = nn.Conv1d(
            in_ch,
            out_ch,
            kernel_size,
            padding=self.pad,
            dilation=dilation,
            bias=False,
        )

    def forward(
        self,
        x,
    ):

        y = self.conv(x)

        if self.pad:
            y = y[
                ...,
                :-self.pad
            ]

        return y


class TCNResidualBlock(
    nn.Module
):

    def __init__(
        self,
        dim,
        filters=32,
        depth=2,
        kernel_size=4,
        dropout=0.30,
    ):

        super().__init__()

        self.proj = (
            nn.Conv1d(
                dim,
                filters,
                1,
            )
            if dim != filters
            else nn.Identity()
        )

        self.blocks = nn.ModuleList()

        for i in range(depth):

            dilation = 2 ** i

            self.blocks.append(
                nn.ModuleDict({
                    "c1":
                        CausalConv1d(
                            filters,
                            filters,
                            kernel_size,
                            dilation,
                        ),
                    "bn1":
                        nn.BatchNorm1d(
                            filters
                        ),
                    "c2":
                        CausalConv1d(
                            filters,
                            filters,
                            kernel_size,
                            dilation,
                        ),
                    "bn2":
                        nn.BatchNorm1d(
                            filters
                        ),
                    "drop":
                        nn.Dropout(
                            dropout
                        ),
                })
            )

    def forward(
        self,
        x,
    ):

        z = x.transpose(
            1,
            2,
        )

        residual = self.proj(
            z
        )

        for block in self.blocks:

            h = block["c1"](
                residual
            )

            h = F.elu(
                block["bn1"](h)
            )

            h = block["drop"](h)

            h = block["c2"](h)

            h = F.elu(
                block["bn2"](h)
            )

            h = block["drop"](h)

            residual = F.elu(
                residual + h
            )

        return residual.transpose(
            1,
            2,
        )


class ATCNetConvBlock(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        F1=16,
        D=2,
        dropout=0.30,
    ):

        super().__init__()

        F2 = F1 * D

        self.temporal = nn.Conv2d(
            1,
            F1,
            kernel_size=(
                1,
                64,
            ),
            padding=(
                0,
                32,
            ),
            bias=False,
        )

        self.bn1 = nn.BatchNorm2d(
            F1
        )

        self.spatial = nn.Conv2d(
            F1,
            F2,
            kernel_size=(
                n_channels,
                1,
            ),
            groups=F1,
            bias=False,
        )

        self.bn2 = nn.BatchNorm2d(
            F2
        )

        self.pool1 = nn.AvgPool2d(
            (
                1,
                8,
            )
        )

        self.drop1 = nn.Dropout(
            dropout
        )

        self.refine = nn.Conv2d(
            F2,
            F2,
            kernel_size=(
                1,
                16,
            ),
            padding=(
                0,
                8,
            ),
            bias=False,
        )

        self.bn3 = nn.BatchNorm2d(
            F2
        )

        self.pool2 = nn.AvgPool2d(
            (
                1,
                7,
            )
        )

        self.drop2 = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
    ):

        z = x.unsqueeze(
            1
        )

        z = F.elu(
            self.bn1(
                self.temporal(z)
            )
        )

        z = F.elu(
            self.bn2(
                self.spatial(z)
            )
        )

        z = self.drop1(
            self.pool1(z)
        )

        z = F.elu(
            self.bn3(
                self.refine(z)
            )
        )

        z = self.drop2(
            self.pool2(z)
        )

        z = z.squeeze(
            2
        )

        return z.transpose(
            1,
            2,
        )

In [8]:
# ============================================================
# CELL 8 — ATCNET-3C
# ============================================================

class ATCNet3C(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        n_samples=640,
        n_classes=3,
        n_windows=5,
        F1=16,
        D=2,
        attn_heads=2,
        attn_dropout=0.30,
        tcn_filters=32,
        tcn_dropout=0.30,
    ):

        super().__init__()

        self.n_windows = n_windows

        self.conv = (
            ATCNetConvBlock(
                n_channels=n_channels,
                F1=F1,
                D=D,
            )
        )

        self.feature_dim = F1 * D

        self.attn = nn.ModuleList([
            nn.MultiheadAttention(
                self.feature_dim,
                attn_heads,
                dropout=attn_dropout,
                batch_first=True,
            )
            for _ in range(
                n_windows
            )
        ])

        self.norms = nn.ModuleList([
            nn.LayerNorm(
                self.feature_dim
            )
            for _ in range(
                n_windows
            )
        ])

        self.tcn = nn.ModuleList([
            TCNResidualBlock(
                self.feature_dim,
                filters=tcn_filters,
                depth=2,
                kernel_size=4,
                dropout=tcn_dropout,
            )
            for _ in range(
                n_windows
            )
        ])

        self.window_head = nn.ModuleList([
            nn.Sequential(
                nn.Linear(
                    tcn_filters,
                    64,
                ),
                nn.ELU(),
                nn.Dropout(0.25),
                nn.Linear(
                    64,
                    n_classes,
                ),
            )
            for _ in range(
                n_windows
            )
        ])

        self.eval()

        with torch.no_grad():

            dummy = torch.zeros(
                2,
                n_channels,
                n_samples,
            )

            seq = self.conv(
                dummy
            )

            self.seq_len = (
                seq.shape[1]
            )

        if self.seq_len < n_windows:
            raise RuntimeError(
                "ATCNet window geometry invalid."
            )

    def forward(
        self,
        x,
    ):

        z = self.conv(
            x
        )

        logits = []

        for i in range(
            self.n_windows
        ):

            start = i

            end = (
                self.seq_len
                - self.n_windows
                + i
                + 1
            )

            w = z[
                :,
                start:end,
                :
            ]

            a, _ = self.attn[i](
                w,
                w,
                w,
                need_weights=False,
            )

            w = self.norms[i](
                w + a
            )

            w = self.tcn[i](
                w
            )

            logits.append(
                self.window_head[i](
                    w[:, -1, :]
                )
            )

        return torch.stack(
            logits,
            dim=0,
        ).mean(
            dim=0
        )

In [9]:
# ============================================================
# CELL 9 — DAFM-3C
# ============================================================

class DAFM3C(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        hidden_channels=8,
    ):

        super().__init__()

        self.temporal_spatial = nn.Sequential(

            nn.Conv2d(
                1,
                hidden_channels,
                kernel_size=(
                    3,
                    15,
                ),
                padding=(
                    1,
                    7,
                ),
                bias=False,
            ),

            nn.BatchNorm2d(
                hidden_channels
            ),

            nn.ELU(),
        )

        self.spatial = nn.Conv2d(
            hidden_channels,
            hidden_channels,
            kernel_size=(
                n_channels,
                1,
            ),
            groups=hidden_channels,
            bias=False,
        )

        self.bn = nn.BatchNorm2d(
            hidden_channels
        )

        self.temporal_attention = nn.Conv2d(
            hidden_channels,
            1,
            kernel_size=(
                1,
                15,
            ),
            padding=(
                0,
                7,
            ),
        )

        self.channel_attention = nn.Conv2d(
            hidden_channels,
            hidden_channels,
            kernel_size=(
                1,
                1,
            ),
        )

        self.out = nn.Conv2d(
            hidden_channels,
            hidden_channels,
            kernel_size=(
                1,
                7,
            ),
            padding=(
                0,
                3,
            ),
            bias=False,
        )

        self.out_bn = nn.BatchNorm2d(
            hidden_channels
        )

        self.pool = nn.AvgPool2d(
            (
                1,
                4,
            )
        )

        self.drop = nn.Dropout(
            0.30
        )

        # Feature classifier.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(64),
            nn.ELU(),
            nn.Dropout(0.35),
            nn.Linear(
                64,
                3,
            ),
        )

    def forward(
        self,
        x,
    ):

        z = x.unsqueeze(
            1
        )

        z = self.temporal_spatial(
            z
        )

        z = F.elu(
            self.bn(
                self.spatial(z)
            )
        )

        # Temporal attention.
        t = torch.sigmoid(
            self.temporal_attention(
                z
            )
        )

        # Channel attention.
        ch = torch.sigmoid(
            self.channel_attention(
                z
            )
        )

        z = (
            z
            * (
                1.0
                + t
            )
            * (
                1.0
                + ch
            )
        )

        z = F.elu(
            self.out_bn(
                self.out(z)
            )
        )

        z = self.pool(
            z
        )

        z = self.drop(
            z
        )

        return self.classifier(
            z
        )


# ------------------------------------------------------------
# Smoke forward
# ------------------------------------------------------------

_dafm_test = DAFM3C().to(
    device
)

with torch.no_grad():

    out = _dafm_test(
        torch.randn(
            2,
            22,
            640,
            device=device,
        )
    )

print(
    "DAFM output:",
    out.shape,
)

assert tuple(
    out.shape
) == (
    2,
    3,
)

del _dafm_test
gc.collect()

DAFM output: torch.Size([2, 3])


0

In [10]:
# ============================================================
# CELL 10 — LOADERS / AUGMENTATION
# ============================================================

def augment_eeg(
    x,
):

    x = x.clone()

    B, C, T = x.shape

    if torch.rand(
        1,
        device=x.device,
    ).item() < 0.35:

        x *= torch.empty(
            B,
            1,
            1,
            device=x.device,
        ).uniform_(
            0.92,
            1.08,
        )

    if torch.rand(
        1,
        device=x.device,
    ).item() < 0.20:

        x += (
            0.004
            * torch.randn_like(
                x
            )
        )

    return x


def make_train_loader(
    X,
    y,
    batch_size=64,
):

    y = np.asarray(
        y,
        dtype=np.int64,
    )

    ds = TensorDataset(
        torch.from_numpy(
            X.astype(
                np.float32
            )
        ),
        torch.from_numpy(
            y
        ),
    )

    counts = np.bincount(
        y,
        minlength=3,
    ).astype(
        np.float64
    )

    inv = np.zeros(
        3,
        dtype=np.float64,
    )

    valid = counts > 0

    inv[valid] = (
        1.0
        / counts[valid]
    )

    sampler = WeightedRandomSampler(
        torch.as_tensor(
            inv[y],
            dtype=torch.double,
        ),
        num_samples=len(y),
        replacement=True,
    )

    return DataLoader(
        ds,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=0,
    )


def make_eval_loader(
    X,
    batch_size=128,
):

    ds = TensorDataset(
        torch.from_numpy(
            np.asarray(
                X,
                dtype=np.float32,
            )
        ),
        torch.zeros(
            len(X),
            dtype=torch.long,
        ),
    )

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

In [11]:
# ============================================================
# CELL 11 — GENERIC TRAIN / PREDICT
# ============================================================

@torch.no_grad()
def predict_probs(
    model,
    X,
):

    model.eval()

    outputs = []

    for xb, _ in make_eval_loader(
        X
    ):

        xb = xb.to(
            device
        )

        logits = model(
            xb
        )

        outputs.append(
            logits.detach()
            .cpu()
            .numpy()
        )

    logits = np.concatenate(
        outputs,
        axis=0,
    )

    logits -= logits.max(
        axis=1,
        keepdims=True,
    )

    P = np.exp(
        logits
    )

    P /= (
        P.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return P.astype(
        np.float32
    )


def train_model(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    seed,
    epochs=120,
    lr=7e-4,
    patience=25,
):

    seed_everything(
        seed
    )

    model = model.to(
        device
    )

    counts = np.bincount(
        y_train,
        minlength=3,
    ).astype(
        np.float32
    )

    weights = (
        counts.sum()
        /
        (
            3.0
            * np.maximum(
                counts,
                1.0,
            )
        )
    )

    weights /= (
        weights.mean()
        + 1e-12
    )

    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(
            weights,
            dtype=torch.float32,
            device=device,
        ),
        label_smoothing=0.01,
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=2e-4,
    )

    scheduler = (
        optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.80,
            patience=8,
            min_lr=1e-5,
        )
    )

    loader = make_train_loader(
        X_train,
        y_train,
        batch_size=64,
    )

    best_state = None
    best_loss = np.inf
    best_epoch = 0
    best_bacc = -np.inf
    wait = 0

    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        losses = []

        for xb, yb in loader:

            xb = xb.to(
                device
            )

            yb = yb.to(
                device
            )

            xb = augment_eeg(
                xb
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                xb
            )

            loss = criterion(
                logits,
                yb,
            )

            if not torch.isfinite(
                loss
            ):
                continue

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=3.0,
            )

            optimizer.step()

            losses.append(
                float(
                    loss.item()
                )
            )

        P_val = predict_probs(
            model,
            X_val,
        )

        pred_val = P_val.argmax(
            axis=1
        )

        val_loss = -float(
            np.mean(
                np.log(
                    np.clip(
                        P_val[
                            np.arange(
                                len(y_val)
                            ),
                            y_val,
                        ],
                        1e-8,
                        1.0,
                    )
                )
            )
        )

        val_acc = (
            accuracy_score(
                y_val,
                pred_val,
            )
            * 100.0
        )

        val_bacc = (
            balanced_accuracy_score(
                y_val,
                pred_val,
            )
            * 100.0
        )

        scheduler.step(
            val_loss
        )

        history.append({
            "epoch": epoch,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_bacc": val_bacc,
        })

        if val_loss < (
            best_loss - 1e-5
        ):

            best_loss = val_loss
            best_bacc = val_bacc
            best_epoch = epoch
            wait = 0

            best_state = copy.deepcopy(
                model.state_dict()
            )

        else:

            wait += 1

        if (
            epoch == 1
            or epoch % 10 == 0
        ):

            print(
                f"    epoch {epoch:03d} | "
                f"val={val_acc:5.1f}% | "
                f"bAcc={val_bacc:5.1f}% | "
                f"vLoss={val_loss:.4f}"
            )

        if wait >= patience:

            print(
                f"    early stop at "
                f"{epoch}; best={best_epoch}"
            )

            break

    if best_state is None:

        raise RuntimeError(
            "No checkpoint."
        )

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(
            history
        ),
        best_epoch,
        best_loss,
        best_bacc,
    )

In [12]:
# ============================================================
# CELL 12 — SOURCE DATA + TARGET NORMALIZATION + FUSION
# ============================================================

def get_fold_data(
    target_subject,
):

    target_subject = str(
        target_subject
    )

    source_mask = (
        bci_meta["subject"].astype(str)
        != target_subject
    )

    target_mask = (
        bci_meta["subject"].astype(str)
        == target_subject
    )

    source_idx = (
        bci_meta.loc[
            source_mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    target_idx = (
        bci_meta.loc[
            target_mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    X_source_raw = load_indices(
        source_idx
    )

    X_target_raw = load_indices(
        target_idx
    )

    y_source = (
        bci_meta.loc[
            source_mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    y_target = (
        bci_meta.loc[
            target_mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    # Source normalization.
    source_norm = (
        RobustNormalizer()
        .fit(
            X_source_raw
        )
    )

    X_source = source_norm.transform(
        X_source_raw
    )

    # --------------------------------------------------------
    # TARGET-SPECIFIC NORMALIZATION
    #
    # Uses target EEG values but not labels.
    # --------------------------------------------------------

    X_target, target_norm = (
        fit_target_normalizer(
            X_target_raw
        )
    )

    # --------------------------------------------------------
    # Source validation split.
    # --------------------------------------------------------

    train_idx, val_idx = (
        stratified_split(
            y_source,
            val_fraction=0.20,
            seed=SEED,
        )
    )

    return {
        "X_train": X_source[train_idx],
        "y_train": y_source[train_idx],
        "X_val": X_source[val_idx],
        "y_val": y_source[val_idx],
        "X_target": X_target,
        "y_target": y_target,
        "source_norm": source_norm,
        "target_norm": target_norm,
    }


def choose_fusion_weight(
    y_val,
    P_atc,
    P_dafm,
):

    candidates = np.arange(
        0.0,
        1.01,
        0.05,
    )

    rows = []

    for alpha in candidates:

        P = (
            alpha
            * P_atc
            +
            (
                1.0
                - alpha
            )
            * P_dafm
        )

        pred = P.argmax(
            axis=1
        )

        acc = (
            accuracy_score(
                y_val,
                pred,
            )
            * 100.0
        )

        bacc = (
            balanced_accuracy_score(
                y_val,
                pred,
            )
            * 100.0
        )

        rows.append({
            "alpha_atcnet":
                float(alpha),
            "alpha_dafm":
                float(
                    1.0 - alpha
                ),
            "accuracy":
                acc,
            "bacc":
                bacc,
        })

    df = pd.DataFrame(
        rows
    )

    # Primary criterion: bAcc.
    # Secondary criterion: accuracy.
    df = (
        df
        .sort_values(
            [
                "bacc",
                "accuracy",
            ],
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )

    best = df.iloc[0]

    return (
        float(
            best[
                "alpha_atcnet"
            ]
        ),
        df,
    )


def run_fold(
    target_subject,
    fold_id,
):

    t0 = time.time()

    d = get_fold_data(
        target_subject
    )

    print(
        "\n"
        + "=" * 78
    )

    print(
        f"ATCNet + DAFM "
        f"LOSO [{fold_id}/9] — {target_subject}"
    )

    print(
        "=" * 78
    )

    print(
        "Train:",
        d["X_train"].shape
    )

    print(
        "Val:",
        d["X_val"].shape
    )

    print(
        "Target:",
        d["X_target"].shape
    )

    # --------------------------------------------------------
    # ATCNet seed 42
    # --------------------------------------------------------

    atc42, _, _, _, _ = (
        train_model(
            ATCNet3C(),
            d["X_train"],
            d["y_train"],
            d["X_val"],
            d["y_val"],
            seed=42,
            epochs=120,
            lr=9e-4,
            patience=25,
        )
    )

    # --------------------------------------------------------
    # ATCNet seed 123
    # --------------------------------------------------------

    atc123, _, _, _, _ = (
        train_model(
            ATCNet3C(),
            d["X_train"],
            d["y_train"],
            d["X_val"],
            d["y_val"],
            seed=123,
            epochs=120,
            lr=9e-4,
            patience=25,
        )
    )

    # --------------------------------------------------------
    # DAFM seed 42
    # --------------------------------------------------------

    dafm42, _, _, _, _ = (
        train_model(
            DAFM3C(),
            d["X_train"],
            d["y_train"],
            d["X_val"],
            d["y_val"],
            seed=42,
            epochs=120,
            lr=7e-4,
            patience=20,
        )
    )

    # --------------------------------------------------------
    # DAFM seed 123
    # --------------------------------------------------------

    dafm123, _, _, _, _ = (
        train_model(
            DAFM3C(),
            d["X_train"],
            d["y_train"],
            d["X_val"],
            d["y_val"],
            seed=123,
            epochs=120,
            lr=7e-4,
            patience=20,
        )
    )

    # --------------------------------------------------------
    # Model-family validation ensembles
    # --------------------------------------------------------

    P_atc_val = (
        0.5
        * (
            predict_probs(
                atc42,
                d["X_val"],
            )
            +
            predict_probs(
                atc123,
                d["X_val"],
            )
        )
    )

    P_dafm_val = (
        0.5
        * (
            predict_probs(
                dafm42,
                d["X_val"],
            )
            +
            predict_probs(
                dafm123,
                d["X_val"],
            )
        )
    )

    alpha, fusion_table = (
        choose_fusion_weight(
            d["y_val"],
            P_atc_val,
            P_dafm_val,
        )
    )

    print(
        "\nBest source-validation fusion:"
    )

    display(
        fusion_table.head(5)
    )

    print(
        f"ATCNet weight = {alpha:.2f}"
    )

    print(
        f"DAFM weight   = {1.0-alpha:.2f}"
    )

    # --------------------------------------------------------
    # Target test predictions.
    # --------------------------------------------------------

    P_atc_test = (
        0.5
        * (
            predict_probs(
                atc42,
                d["X_target"],
            )
            +
            predict_probs(
                atc123,
                d["X_target"],
            )
        )
    )

    P_dafm_test = (
        0.5
        * (
            predict_probs(
                dafm42,
                d["X_target"],
            )
            +
            predict_probs(
                dafm123,
                d["X_target"],
            )
        )
    )

    # --------------------------------------------------------
    # Separate results.
    # --------------------------------------------------------

    pred_atc = P_atc_test.argmax(
        axis=1
    )

    pred_dafm = P_dafm_test.argmax(
        axis=1
    )

    atc_acc = (
        accuracy_score(
            d["y_target"],
            pred_atc,
        )
        * 100.0
    )

    dafm_acc = (
        accuracy_score(
            d["y_target"],
            pred_dafm,
        )
        * 100.0
    )

    # --------------------------------------------------------
    # Final fixed fusion.
    # --------------------------------------------------------

    P_fused = (
        alpha
        * P_atc_test
        +
        (
            1.0
            - alpha
        )
        * P_dafm_test
    )

    pred_fused = P_fused.argmax(
        axis=1
    )

    fused_acc = (
        accuracy_score(
            d["y_target"],
            pred_fused,
        )
        * 100.0
    )

    fused_bacc = (
        balanced_accuracy_score(
            d["y_target"],
            pred_fused,
        )
        * 100.0
    )

    fused_kappa = (
        cohen_kappa_score(
            d["y_target"],
            pred_fused,
        )
    )

    print(
        "\n"
        + "-" * 78
    )

    print(
        f"Target subject    : "
        f"{target_subject}"
    )

    print(
        f"ATCNet test       : "
        f"{atc_acc:.2f}%"
    )

    print(
        f"DAFM test         : "
        f"{dafm_acc:.2f}%"
    )

    print(
        f"Fused test        : "
        f"{fused_acc:.2f}%"
    )

    print(
        f"Fused bAcc        : "
        f"{fused_bacc:.2f}%"
    )

    print(
        f"Fused kappa       : "
        f"{fused_kappa:.4f}"
    )

    print(
        f"Elapsed           : "
        f"{(time.time()-t0)/60.0:.1f} min"
    )

    print(
        "-" * 78
    )

    return {
        "subject":
            target_subject,
        "atc_acc":
            atc_acc,
        "dafm_acc":
            dafm_acc,
        "fused_acc":
            fused_acc,
        "fused_bacc":
            fused_bacc,
        "kappa":
            fused_kappa,
        "alpha":
            alpha,
        "y_test":
            d["y_target"],
        "pred_fused":
            pred_fused,
        "P_fused":
            P_fused,
    }

In [13]:
# ============================================================
# CELL 13 — S01 SMOKE TEST
# ============================================================

smoke_result = run_fold(
    target_subject="S01",
    fold_id=1,
)

print(
    "\n"
    + "=" * 78
)

print(
    "S01 SMOKE SUMMARY"
)

print(
    "=" * 78
)

print(
    f"ATCNet : "
    f"{smoke_result['atc_acc']:.2f}%"
)

print(
    f"DAFM   : "
    f"{smoke_result['dafm_acc']:.2f}%"
)

print(
    f"FUSED  : "
    f"{smoke_result['fused_acc']:.2f}%"
)

print(
    f"Fused bAcc: "
    f"{smoke_result['fused_bacc']:.2f}%"
)

print(
    f"Fusion alpha (ATCNet): "
    f"{smoke_result['alpha']:.2f}"
)


ATCNet + DAFM LOSO [1/9] — S01
Train: (1383, 22, 640)
Val: (345, 22, 640)
Target: (216, 22, 640)
    epoch 001 | val= 41.4% | bAcc= 41.4% | vLoss=1.0656
    epoch 010 | val= 62.0% | bAcc= 62.0% | vLoss=0.8226
    epoch 020 | val= 62.9% | bAcc= 62.9% | vLoss=0.8255
    epoch 030 | val= 67.8% | bAcc= 67.8% | vLoss=0.8148
    epoch 040 | val= 67.8% | bAcc= 67.8% | vLoss=0.8627
    early stop at 49; best=24
    epoch 001 | val= 42.6% | bAcc= 42.6% | vLoss=1.0643
    epoch 010 | val= 55.4% | bAcc= 55.4% | vLoss=0.9370
    epoch 020 | val= 66.1% | bAcc= 66.1% | vLoss=0.7629
    epoch 030 | val= 66.1% | bAcc= 66.1% | vLoss=0.8815
    epoch 040 | val= 65.2% | bAcc= 65.2% | vLoss=0.9429
    early stop at 46; best=21
    epoch 001 | val= 38.8% | bAcc= 38.8% | vLoss=1.0906
    epoch 010 | val= 48.4% | bAcc= 48.4% | vLoss=1.1171
    epoch 020 | val= 48.4% | bAcc= 48.4% | vLoss=1.2847
    early stop at 23; best=3
    epoch 001 | val= 38.8% | bAcc= 38.8% | vLoss=1.0909
    epoch 010 | val= 42.9% | 

,alpha_atcnet,alpha_dafm,accuracy,bacc
0,0.50,0.50,70.434783,70.434783
1,0.55,0.45,70.434783,70.434783
2,0.60,0.40,70.434783,70.434783
3,0.65,0.35,70.144928,70.144928
4,0.70,0.30,69.565217,69.565217


ATCNet weight = 0.50
DAFM weight   = 0.50

------------------------------------------------------------------------------
Target subject    : S01
ATCNet test       : 74.07%
DAFM test         : 41.20%
Fused test        : 72.69%
Fused bAcc        : 72.69%
Fused kappa       : 0.5903
Elapsed           : 10.6 min
------------------------------------------------------------------------------

S01 SMOKE SUMMARY
ATCNet : 74.07%
DAFM   : 41.20%
FUSED  : 72.69%
Fused bAcc: 72.69%
Fusion alpha (ATCNet): 0.50


In [14]:
# ============================================================
# CELL 14 — FULL 9-SUBJECT LOSO
# ============================================================

RUN_FULL_LOSO = True

if RUN_FULL_LOSO:

    rows = []
    fold_objects = {}

    for fold_id, subject in enumerate(
        BCI_SUBJECTS,
        start=1,
    ):

        result = run_fold(
            target_subject=subject,
            fold_id=fold_id,
        )

        fold_objects[
            subject
        ] = result

        rows.append({
            "fold":
                fold_id,
            "subject":
                subject,
            "atcnet_accuracy":
                result["atc_acc"],
            "dafm_accuracy":
                result["dafm_acc"],
            "fused_accuracy":
                result["fused_acc"],
            "fused_bacc":
                result["fused_bacc"],
            "kappa":
                result["kappa"],
            "alpha_atcnet":
                result["alpha"],
        })

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(
        rows
    )

    print(
        "\n"
        + "=" * 78
    )

    print(
        "FINAL 9-SUBJECT LOSO"
    )

    print(
        "=" * 78
    )

    display(
        results_df
    )

    print(
        "\nATCNet mean:",
        f"{results_df['atcnet_accuracy'].mean():.2f}%"
    )

    print(
        "DAFM mean:",
        f"{results_df['dafm_accuracy'].mean():.2f}%"
    )

    print(
        "Fused mean:",
        f"{results_df['fused_accuracy'].mean():.2f}%"
    )

    print(
        "Fused mean bAcc:",
        f"{results_df['fused_bacc'].mean():.2f}%"
    )

    print(
        "Fused median:",
        f"{results_df['fused_accuracy'].median():.2f}%"
    )

    print(
        "Fused std:",
        f"{results_df['fused_accuracy'].std():.2f}%"
    )

    print(
        "Fused subjects >=70:",
        int(
            (
                results_df[
                    "fused_accuracy"
                ]
                >= 70.0
            ).sum()
        ),
        "/9",
    )

    print(
        "Fused subjects >=80:",
        int(
            (
                results_df[
                    "fused_accuracy"
                ]
                >= 80.0
            ).sum()
        ),
        "/9",
    )

    mean_fused = (
        results_df[
            "fused_accuracy"
        ].mean()
    )

    print(
        "\n"
        + "=" * 78
    )

    if mean_fused >= 80.0:

        print(
            f"✅ 80% TARGET: "
            f"{mean_fused:.2f}%"
        )

    elif mean_fused >= 70.0:

        print(
            f"✅ 70% TARGET: "
            f"{mean_fused:.2f}%"
        )

    else:

        print(
            f"❌ TARGET NOT REACHED: "
            f"{mean_fused:.2f}%"
        )

    print(
        "=" * 78
    )


ATCNet + DAFM LOSO [1/9] — S01
Train: (1383, 22, 640)
Val: (345, 22, 640)
Target: (216, 22, 640)
    epoch 001 | val= 51.3% | bAcc= 51.3% | vLoss=1.0274
    epoch 010 | val= 61.7% | bAcc= 61.7% | vLoss=0.8160
    epoch 020 | val= 63.2% | bAcc= 63.2% | vLoss=0.8250
    epoch 030 | val= 64.3% | bAcc= 64.3% | vLoss=0.8606
    early stop at 37; best=12
    epoch 001 | val= 48.4% | bAcc= 48.4% | vLoss=1.0459
    epoch 010 | val= 60.3% | bAcc= 60.3% | vLoss=0.8667
    epoch 020 | val= 62.0% | bAcc= 62.0% | vLoss=0.8285
    epoch 030 | val= 65.2% | bAcc= 65.2% | vLoss=0.9180
    early stop at 39; best=14
    epoch 001 | val= 38.6% | bAcc= 38.6% | vLoss=1.1037
    epoch 010 | val= 47.8% | bAcc= 47.8% | vLoss=1.1811
    epoch 020 | val= 48.1% | bAcc= 48.1% | vLoss=1.3266
    early stop at 22; best=2
    epoch 001 | val= 35.4% | bAcc= 35.4% | vLoss=1.1067
    epoch 010 | val= 50.1% | bAcc= 50.1% | vLoss=1.0927
    epoch 020 | val= 50.1% | bAcc= 50.1% | vLoss=1.2452
    early stop at 26; best=6


,alpha_atcnet,alpha_dafm,accuracy,bacc
0,0.65,0.35,66.666667,66.666667
1,0.50,0.50,66.086957,66.086957
2,0.80,0.20,66.086957,66.086957
3,0.45,0.55,66.086957,66.086957
4,0.55,0.45,65.797101,65.797101


ATCNet weight = 0.65
DAFM weight   = 0.35

------------------------------------------------------------------------------
Target subject    : S01
ATCNet test       : 60.19%
DAFM test         : 50.00%
Fused test        : 63.43%
Fused bAcc        : 63.43%
Fused kappa       : 0.4514
Elapsed           : 9.3 min
------------------------------------------------------------------------------

ATCNet + DAFM LOSO [2/9] — S02
Train: (1383, 22, 640)
Val: (345, 22, 640)
Target: (216, 22, 640)
    epoch 001 | val= 41.7% | bAcc= 41.7% | vLoss=1.0637
    epoch 010 | val= 63.8% | bAcc= 63.8% | vLoss=0.7608
    epoch 020 | val= 66.4% | bAcc= 66.4% | vLoss=0.7732
    epoch 030 | val= 66.4% | bAcc= 66.4% | vLoss=0.7887
    early stop at 34; best=9
    epoch 001 | val= 46.4% | bAcc= 46.4% | vLoss=1.0391
    epoch 010 | val= 62.3% | bAcc= 62.3% | vLoss=0.7812
    epoch 020 | val= 68.7% | bAcc= 68.7% | vLoss=0.7032
    epoch 030 | val= 69.0% | bAcc= 69.0% | vLoss=0.8114
    epoch 040 | val= 68.1% | bAcc= 68

,alpha_atcnet,alpha_dafm,accuracy,bacc
0,0.65,0.35,71.594203,71.594203
1,0.80,0.20,71.304348,71.304348
2,0.70,0.30,71.014493,71.014493
3,0.75,0.25,71.014493,71.014493
4,1.00,0.00,71.014493,71.014493


ATCNet weight = 0.65
DAFM weight   = 0.35

------------------------------------------------------------------------------
Target subject    : S02
ATCNet test       : 40.28%
DAFM test         : 37.50%
Fused test        : 38.89%
Fused bAcc        : 38.89%
Fused kappa       : 0.0833
Elapsed           : 10.4 min
------------------------------------------------------------------------------

ATCNet + DAFM LOSO [3/9] — S03
Train: (1383, 22, 640)
Val: (345, 22, 640)
Target: (216, 22, 640)
    epoch 001 | val= 42.3% | bAcc= 42.3% | vLoss=1.0740
    epoch 010 | val= 60.9% | bAcc= 60.9% | vLoss=0.8553
    epoch 020 | val= 61.2% | bAcc= 61.2% | vLoss=0.8876


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 15 — CONFUSION MATRIX / SAVE
# ============================================================

if (
    "fold_objects" in globals()
    and len(fold_objects) > 0
):

    y_all = np.concatenate([
        fold_objects[s]["y_test"]
        for s in BCI_SUBJECTS
    ])

    pred_all = np.concatenate([
        fold_objects[s]["pred_fused"]
        for s in BCI_SUBJECTS
    ])

    cm = confusion_matrix(
        y_all,
        pred_all,
        labels=[
            0,1,2
        ],
        normalize="true",
    )

    display(
        pd.DataFrame(
            cm,
            index=CLASSES,
            columns=CLASSES,
        ).round(3)
    )

    print(
        classification_report(
            y_all,
            pred_all,
            labels=[
                0,1,2
            ],
            target_names=CLASSES,
            digits=4,
        )
    )

    if "results_df" in globals():

        save_path = (
            RESULT_DIR
            / "atcnet_dafm_targetnorm_loso.csv"
        )

        results_df.to_csv(
            save_path,
            index=False,
        )

        print(
            "Saved:",
            save_path,
        )

## Interpretation

Report the strict source-trained ATCNet and DAFM scores separately from the fused transductive score.

The fused model uses a fusion coefficient chosen from source validation predictions only. Target labels are not used to select the fusion weight.

Target-specific normalization uses target EEG values without labels. Therefore, the fused result is an **unsupervised/transductive adaptation** result rather than strictly untouched-target LOSO.